# Multilingual Sentiment Analyzer - AWS Marketplace

Deploys **Multilingual Sentiment Analyzer** from AWS Marketplace as a SageMaker endpoint inside **your own AWS account**. Your data never leaves your VPC and there are no external API calls or token limits.

Multilingual sentiment model covering English, Dutch, German, French, Spanish, and Italian. Fine-tuned on product reviews.

## Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.m5.xlarge"  # recommended real-time instance
ENDPOINT_NAME = "multilingual-sentiment-6lang"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Takes roughly 6-9 minutes. The endpoint bills per hour while it exists, so do not skip section 5.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Classify text

The endpoint accepts `application/json` shaped `{"inputs": "..."}`.
Returns `{"label": "positive", "score": 0.99}` — sentiment label (positive/negative/neutral) and confidence.

Pass text in any of the 6 supported languages. The model auto-detects language and applies language-specific features.

In [ ]:
runtime = boto3.client("sagemaker-runtime")


def classify(text):
    """Return label and confidence score for the input text."""
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": text}),
    )
    return json.loads(response["Body"].read())


texts = [
    "Revenue growth exceeded expectations driven by strong cloud adoption.",
    "The company reported a significant loss due to supply chain disruptions.",
    "Quarterly results were in line with analyst estimates.",
]

for text in texts:
    result = classify(text)
    print(f"{result['label']:10s} ({result['score']:.3f})  {text}")

## 4. Batch transform for offline workloads

For processing large datasets without a live endpoint, batch transform avoids paying for an always-on endpoint. Input is JSON Lines, one `{"inputs": "..."}` object per line.

In [ ]:
# transformer = model.transformer(
#     instance_count=1,
#     instance_type=INSTANCE_TYPE,
#     output_path=f"s3://{session.default_bucket()}/multilingual-sentiment-6lang/",
#     strategy="SingleRecord",
# )
# transformer.transform(
#     data=f"s3://{session.default_bucket()}/multilingual-sentiment-6lang-input/",
#     content_type="application/json",
# )
# transformer.wait()

## 5. Clean up

Delete the endpoint when you are done. It bills for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)